In [ ]:
import duckdb
import leafmap
import pandas as pd

## Sample data 

In [ ]:
url = "https://open.gishub.org/data/duckdb/nyc_data.db.zip"
leafmap.download_file(url, unzip=True)

# Connect to the database

In [ ]:
con = duckdb.connect("nyc_data.db")

In [ ]:
con.install_extension("spatial")
con.load_extension("spatial")

In [ ]:
con.sql("SHOW TABLES;")

In [ ]:
con.sql("FROM nyc_subway_stations;")

In [ ]:
con.sql("""

CREATE or REPLACE TABLE samples (name VARCHAR, geom GEOMETRY);

INSERT INTO samples VALUES
  ('Point', ST_GeomFromText('POINT(-100 40)')),
  ('Linestring', ST_GeomFromText('LINESTRING(0 0, 1 1, 2 1, 2 2)')),
  ('Polygon', ST_GeomFromText('POLYGON((0 0, 1 0, 1 1, 0 1, 0 0))')),
  ('PolygonWithHole', ST_GeomFromText('POLYGON((0 0, 10 0, 10 10, 0 10, 0 0),(1 1, 1 2, 2 2, 2 1, 1 1))')),
  ('Collection', ST_GeomFromText('GEOMETRYCOLLECTION(POINT(2 0),POLYGON((0 0, 1 0, 1 1, 0 1, 0 0)))'));

SELECT * FROM samples;
          
  """)

In [ ]:
con.sql("""

CREATE or REPLACE TABLE samples (name VARCHAR, geom GEOMETRY);

INSERT INTO samples VALUES
  ('q', ST_GeomFromText('POINT(-100 40)')),
  ('s', ST_GeomFromText('LINESTRING(0 0, 1 1, 2 1, 2 2)')),
  ('Polygon', ST_GeomFromText('POLYGON((0 0, 1 0, 1 1, 0 1, 0 0))')),
  ('PolygonWithHole', ST_GeomFromText('POLYGON((0 0, 10 0, 10 10, 0 10, 0 0),(1 1, 1 2, 2 2, 2 1, 1 1))')),
  ('Collection', ST_GeomFromText('GEOMETRYCOLLECTION(POINT(2 0),POLYGON((0 0, 1 0, 1 1, 0 1, 0 0)))'));

SELECT * FROM samples;
          
  """)

In [ ]:
con.sql("""
        
COPY samples TO 'samples.geojson' (FORMAT GDAL, DRIVER GeoJSON);
        
""")

In [ ]:
import geopandas as gpd

In [ ]:
gdf = gpd.read_file("samples.geojson")
gdf

In [ ]:
gdf.explore()

In [ ]:
con.sql("""

SELECT ST_AsText(geom)
  FROM samples
  WHERE name = 'Point';
        
""")

In [ ]:
con.sql("""

SELECT ST_X(geom) as longitude, ST_Y(geom) as latitude
  FROM samples
  WHERE name = 'Point';
        
""")

In [ ]:
con.sql("""

SELECT * FROM nyc_subway_stations
        
""")

In [ ]:
con.sql("""

SELECT * EXCLUDE geom, ST_X(geom) as longitude, ST_Y(geom) as latitude FROM nyc_subway_stations
        
""")

In [ ]:
con.sql("""

SELECT name, ST_AsText(geom)
  FROM nyc_subway_stations
  LIMIT 10;
        
""")

In [ ]:
con.sql("""
        
SELECT ST_AsText(geom)
  FROM samples
  WHERE name = 'Linestring';
        
""")

In [ ]:
con.sql("""

SELECT * EXCLUDE geom, ST_Length(geom)
  FROM nyc_streets
        
""")

In [ ]:
con.sql("""
        
SELECT name, ST_Area(geom)
  FROM samples
  WHERE name LIKE 'Polygon%';
        
""")

# Data Visualization

In [ ]:
con.sql("SHOW TABLES;")

In [ ]:
subway_stations_df = con.sql(
    "SELECT * EXCLUDE geom, ST_AsText(geom) as geometry FROM nyc_subway_stations"
).df()
subway_stations_df.head()

In [ ]:
subway_stations_gdf = leafmap.df_to_gdf(
    subway_stations_df, src_crs="EPSG:26918", dst_crs="EPSG:4326"
)
subway_stations_gdf.head()

In [ ]:
subway_stations_gdf.explore()

In [ ]:
nyc_streets_df = con.sql(
    "SELECT * EXCLUDE geom, ST_AsText(geom) as geometry FROM nyc_streets"
).df()
nyc_streets_df.head()

In [ ]:
nyc_streets_gdf = leafmap.df_to_gdf(
    nyc_streets_df, src_crs="EPSG:26918", dst_crs="EPSG:4326"
)
nyc_streets_gdf.head()

In [ ]:
nyc_streets_gdf.explore()

In [ ]:
con.close()